In [12]:
import ee
ee.Initialize(project="ee-mnolasco")

CARPETA = "users/mnolasco/FORMAR_2024_2026/mascaras_modis"
try:
    ee.data.createAsset({"type": "FOLDER"}, CARPETA)
except Exception as e:
    print("Carpeta:", str(e)[:90])          # si ya existe, avisa y sigue

zona = ee.FeatureCollection("users/mnolasco/IG-SANCO/area_sancor").filter(ee.Filter.eq("Zona", "ARG"))
dem = ee.Image("CGIAR/SRTM90_V4").select("elevation").lt(1000)
modis = ee.ImageCollection("MODIS/061/MOD13Q1").select("NDVI")

# Nolasco et al. (2023), Tabla 4 — NDVI x 10000
# (latitud norte, latitud sur): (emergencia <, floración >, madurez <)
FRANJAS = {(-20, -25): (3500, 8200, 4900),
           (-25, -30): (3500, 7200, 5200),
           (-30, -35): (3300, 7600, 3800),
           (-35, -40): (3800, 7100, 4300)}

def mascara_campania(anio_siembra):
    """Máscara de cultivo estival anual para la campaña anio_siembra/anio_siembra+1."""
    s, e = str(anio_siembra), str(anio_siembra + 1)
    ndvi_min_emer = modis.filterDate(s + "-08-28", e + "-02-09").min()
    ndvi_max_flor = modis.filterDate(s + "-11-14", e + "-04-23").max()
    ndvi_min_mad = modis.filterDate(e + "-02-02", e + "-06-19").min()
    partes = []
    for (lat_n, lat_s), (u_em, u_fl, u_ma) in FRANJAS.items():
        franja = ee.Geometry.BBox(-70, lat_s, -50, lat_n).intersection(zona.geometry(), 10)
        cumple = ndvi_min_emer.lt(u_em).And(ndvi_max_flor.gt(u_fl)).And(ndvi_min_mad.lt(u_ma))
        partes.append(cumple.clip(franja))
    return ee.ImageCollection(partes).mosaic().And(dem).rename("agricola").toByte().clip(zona)

# --- control rápido: superficie de la campaña 2020/21 (comparala con tu mapa de Drive) ---
km2 = (mascara_campania(2020).multiply(ee.Image.pixelArea())
       .reduceRegion(ee.Reducer.sum(), zona.geometry(), scale=250, maxPixels=1e10)
       .getNumber("agricola").divide(1e6).getInfo())
print(f"Campaña 2020/21: {km2:,.0f} km² de cultivo estival anual")

Carpeta: Cannot overwrite asset 'projects/earthengine-legacy/assets/users/mnolasco/FORMAR_2024_2026
Campaña 2020/21: 279,774 km² de cultivo estival anual


In [13]:
region = zona.geometry().bounds()
for a in range(2000, 2016):                       # 2000/01 … 2015/16
    nombre = f"modis_agricola_{a + 1}"
    t = ee.batch.Export.image.toAsset(
        image=mascara_campania(a), description=nombre, assetId=f"{CARPETA}/{nombre}",
        region=region, scale=250, crs="EPSG:4326",
        pyramidingPolicy={"agricola": "mode"}, maxPixels=1e10)
    t.start()
    print(nombre, t.status()["state"])

modis_agricola_2001 READY
modis_agricola_2002 READY
modis_agricola_2003 READY
modis_agricola_2004 READY
modis_agricola_2005 READY
modis_agricola_2006 READY
modis_agricola_2007 READY
modis_agricola_2008 READY
modis_agricola_2009 READY
modis_agricola_2010 READY
modis_agricola_2011 READY
modis_agricola_2012 READY
modis_agricola_2013 READY
modis_agricola_2014 READY
modis_agricola_2015 READY
modis_agricola_2016 READY


In [1]:
import numpy as np
import rasterio

RUTA = "../data/raw/2020-2021_mascara_cultivos_3M_DEM1000.tif"
FRANJAS_LAT = [(-20, -25), (-25, -30), (-30, -35), (-35, -40)]

with rasterio.open(RUTA) as r:
    a = r.read(1)
    t = r.transform
    print("CRS:", r.crs, "| resolución:", r.res, "| tamaño:", a.shape, "| valores:", np.unique(a)[:10])
    geografico = r.crs.is_geographic

lat_sup = t.f + t.e * np.arange(a.shape[0])          # latitud del borde superior de cada fila
lat_inf = lat_sup + t.e
if geografico:                                       # área de cada píxel según su latitud (km²)
    R = 6371.0088
    area_fila = R ** 2 * np.radians(t.a) * np.abs(np.sin(np.radians(lat_sup)) - np.sin(np.radians(lat_inf)))
else:
    area_fila = np.full(a.shape[0], abs(t.a * t.e) / 1e6)

km2_fila = (a == 1).sum(axis=1) * area_fila
lat_centro = (lat_sup + lat_inf) / 2
print(f"\nMapa original 2020/21: {km2_fila.sum():,.0f} km²")
for n, s in FRANJAS_LAT:
    sel_f = (lat_centro <= n) & (lat_centro > s)
    print(f"  franja {abs(n)}–{abs(s)}° S: {km2_fila[sel_f].sum():,.0f} km²")

CRS: EPSG:4326 | resolución: (0.002245788210298804, 0.002245788210298804) | tamaño: (9099, 8907) | valores: [0 1]

Mapa original 2020/21: 280,901 km²
  franja 20–25° S: 8,172 km²
  franja 25–30° S: 69,064 km²
  franja 30–35° S: 135,308 km²
  franja 35–40° S: 68,357 km²


In [4]:
m = mascara_campania(2020).multiply(ee.Image.pixelArea()).divide(1e6)
print("Mapa recalculado 2020/21 (MODIS 061):")
for (n, s) in FRANJAS:
    g = ee.Geometry.BBox(-70, s, -50, n).intersection(zona.geometry(), 10)
    v = m.reduceRegion(ee.Reducer.sum(), g, scale=250, maxPixels=1e10).getNumber("agricola").getInfo()
    print(f"  franja {abs(n)}–{abs(s)}° S: {v:,.0f} km²")

Mapa recalculado 2020/21 (MODIS 061):
  franja 20–25° S: 8,171 km²
  franja 25–30° S: 69,025 km²
  franja 30–35° S: 134,646 km²
  franja 35–40° S: 67,951 km²


In [5]:
import ee
ee.Initialize(project="ee-mnolasco")
BASE, CRS = "users/mnolasco/FORMAR_2024_2026", "EPSG:32720"
CARPETA = f"{BASE}/mascaras_modis"
EXT = [10, 11, 12, 15, 16, 18]                      # maíz, soja, girasol, algodón, maní, sorgo
CAMPANIAS = range(2020, 2026)
MASCARAS = ["agri19", "modis", "modis_ant", "modis_2de3"]

deptos = ee.FeatureCollection(f"{BASE}/departments_selected").select(["link"])
agri19 = ee.Image(f"{BASE}/AgriLand-2019").eq(1).unmask(0)
modis = lambda y: ee.Image(f"{CARPETA}/modis_agricola_{y}").unmask(0)

bandas = []
for y in CAMPANIAS:
    m = ee.Image(f"{BASE}/MNC_verano-{y}")
    ext = m.remap(EXT, [1] * len(EXT), 0).unmask(0)
    soja = m.eq(11).unmask(0)
    masc = {"agri19": agri19, "modis": modis(y), "modis_ant": modis(y - 1),
            "modis_2de3": modis(y - 1).add(modis(y - 2)).add(modis(y - 3)).gte(2)}
    bandas += [ext.rename(f"ext_{y}"), soja.rename(f"soja_{y}")]
    for k, M in masc.items():
        bandas += [M.rename(f"{k}_{y}"),                          # área de la máscara
                   ext.And(M).rename(f"{k}_ext_{y}"),             # máscara que fue agrícola ese año
                   soja.And(M.Not()).rename(f"{k}_sojafuera_{y}")] # soja que la máscara deja afuera
img = ee.Image.cat(bandas).multiply(ee.Image.pixelArea().divide(1e4))   # hectáreas

def areas(fc):
    return (img.reduceRegions(fc, ee.Reducer.sum(), scale=30, crs=CRS, tileScale=8)
            .map(lambda f: f.setGeometry(f.geometry().centroid(100))))

# --- prueba: Rosario, campaña 2021 --------------------------------------------
p = areas(deptos.filter(ee.Filter.eq("link", "82084"))).first().getInfo()["properties"]
y = 2021
print(f"Rosario {y}: agrícola INTA {p[f'ext_{y}']:,.0f} ha | soja INTA {p[f'soja_{y}']:,.0f} ha")
for k in MASCARAS:
    print(f"  {k:11s} área {p[f'{k}_{y}']:>9,.0f} ha | soja fuera {100 * p[f'{k}_sojafuera_{y}'] / p[f'soja_{y}']:5.1f} % "
          f"| máscara no agrícola {100 * (1 - p[f'{k}_ext_{y}'] / p[f'{k}_{y}']):5.1f} %")

Rosario 2021: agrícola INTA 107,321 ha | soja INTA 74,479 ha
  agri19      área   106,155 ha | soja fuera   8.4 % | máscara no agrícola   7.8 %
  modis       área    71,810 ha | soja fuera  37.1 % | máscara no agrícola   6.3 %
  modis_ant   área    71,009 ha | soja fuera  39.7 % | máscara no agrícola   6.2 %
  modis_2de3  área    73,655 ha | soja fuera  35.6 % | máscara no agrícola   5.2 %


In [6]:
tarea = ee.batch.Export.table.toAsset(
    collection=areas(deptos),
    description="comparacion_mascaras_2020_2025",
    assetId=f"{BASE}/comparacion_mascaras_2020_2025")
tarea.start()
print("Tarea lanzada:", tarea.status()["state"])

Tarea lanzada: READY


In [7]:
import ee
import pandas as pd
ee.Initialize(project="ee-mnolasco")
BASE = "users/mnolasco/FORMAR_2024_2026"
CAMPANIAS = range(2020, 2026)
MASCARAS = ["agri19", "modis", "modis_ant", "modis_2de3"]

# --- 1. Comparación de máscaras contra INTA ---------------------------------------
feats = ee.FeatureCollection(f"{BASE}/comparacion_mascaras_2020_2025").getInfo()["features"]
info = pd.read_csv("../results/tables/seleccion_imagen_2021.csv", dtype={"link": str})[["link", "departamento", "provincia"]]
filas = []
for f in feats:
    p = f["properties"]
    for y in CAMPANIAS:
        for k in MASCARAS:
            filas.append({"link": str(p["link"]).zfill(5), "campania": y, "mascara": k,
                          "ext": p[f"ext_{y}"], "soja": p[f"soja_{y}"], "area": p[f"{k}_{y}"],
                          "ext_en": p[f"{k}_ext_{y}"], "soja_fuera": p[f"{k}_sojafuera_{y}"]})
cm = pd.DataFrame(filas).merge(info, on="link", how="left")
cm.to_csv("../results/tables/comparacion_mascaras_2020_2025.csv", index=False)

def ind(g):
    s = g[["ext", "soja", "area", "ext_en", "soja_fuera"]].sum()
    return pd.Series({"soja_fuera_%": 100 * s["soja_fuera"] / s["soja"],
                      "mascara_no_agri_%": 100 * (1 - s["ext_en"] / s["area"]),
                      "area_vs_INTA_%": 100 * s["area"] / s["ext"]})

print(f"Departamentos: {cm['link'].nunique()} | campañas: {list(CAMPANIAS)}\n")
print("1. Promedio 2020–2025, total de los 95 departamentos:")
print(cm.groupby("mascara").apply(ind).reindex(MASCARAS).round(1).to_string())
print("\n2. Soja fuera de la máscara (%), por campaña:")
print(cm.groupby(["campania", "mascara"]).apply(ind)["soja_fuera_%"].unstack()[MASCARAS].round(1).to_string())
print("\n3. Por provincia (2020–2025): soja fuera (%)")
pp = cm.groupby(["provincia", "mascara"]).apply(ind).unstack("mascara")
print(pp["soja_fuera_%"][MASCARAS].round(1).to_string())
print("\n   Máscara que no fue agrícola (%)")
print(pp["mascara_no_agri_%"][MASCARAS].round(1).to_string())

# --- 2. Máscaras Sentinel-2

Departamentos: 95 | campañas: [2020, 2021, 2022, 2023, 2024, 2025]

1. Promedio 2020–2025, total de los 95 departamentos:
            soja_fuera_%  mascara_no_agri_%  area_vs_INTA_%
mascara                                                    
agri19              23.0               13.5            86.0
modis               29.1               15.0            83.1
modis_ant           31.1               17.0            82.2
modis_2de3          30.3               14.9            81.4

2. Soja fuera de la máscara (%), por campaña:
mascara   agri19  modis  modis_ant  modis_2de3
campania                                      
2020        21.2   28.8       36.9        36.0
2021        24.5   25.2       31.4        35.3
2022        20.6   26.1       24.6        27.4
2023        20.8   36.8       25.8        23.4
2024        24.8   25.1       39.4        28.2
2025        25.2   32.4       27.6        29.9

3. Por provincia (2020–2025): soja fuera (%)
mascara       agri19  modis  modis_ant  modis_2de

In [8]:
s2 = ee.data.listAssets({"parent": "projects/earthengine-legacy/assets/users/mnolasco/IG-SANCOR_BiMap_60m"})
for a in s2.get("assets", []):
    print(a["name"].split("/")[-1])
img = ee.Image("users/mnolasco/IG-SANCOR_BiMap_60m/Bi_map_2021-2022_60m")
print("Bandas:", img.bandNames().getInfo(), "| proyección:", img.projection().crs().getInfo(),
      "| escala:", round(img.projection().nominalScale().getInfo()))

Bi_map_2000-2001_60m
Bi_map_2001-2002_60m
Bi_map_2002-2003_60m
Bi_map_2003-2004_60m
Bi_map_2004-2005_60m
Bi_map_2005-2006_60m
Bi_map_2006-2007_60m
Bi_map_2007-2008_60m
Bi_map_2008-2009_60m
Bi_map_2009-2010_60m
Bi_map_2010-2011_60m
Bi_map_2011-2012_60m
Bi_map_2012-2013_60m
Bi_map_2013-2014_60m
Bi_map_2014-2015_60m
Bi_map_2015-2016_60m
Bi_map_2016-2017_60m
Bi_map_2017-2018_60m
Bi_map_2018-2019_60m
Bi_map_2019-2020_60m
Bi_map_2020-2021_60m
Bi_map_2021-2022_60m
Bandas: ['b1'] | proyección: EPSG:32720 | escala: 60


In [9]:
import ee
ee.Initialize(project="ee-mnolasco")
BASE, CRS = "users/mnolasco/FORMAR_2024_2026", "EPSG:32720"
EXT = [10, 11, 12, 15, 16, 18]
deptos = ee.FeatureCollection(f"{BASE}/departments_selected").select(["link"])
s2m = lambda y: ee.Image(f"users/mnolasco/IG-SANCOR_BiMap_60m/Bi_map_{y - 1}-{y}_60m").eq(1).unmask(0)

bandas = []
for y in [2020, 2021, 2022]:
    m = ee.Image(f"{BASE}/MNC_verano-{y}")
    ext = m.remap(EXT, [1] * len(EXT), 0).unmask(0)
    soja = m.eq(11).unmask(0)
    M = s2m(y)
    bandas += [ext.rename(f"ext_{y}"), soja.rename(f"soja_{y}"),
               M.rename(f"s2_{y}"), ext.And(M).rename(f"s2_ext_{y}"),
               soja.And(M.Not()).rename(f"s2_sojafuera_{y}")]
img_s2 = ee.Image.cat(bandas).multiply(ee.Image.pixelArea().divide(1e4))

def areas_s2(fc):
    return (img_s2.reduceRegions(fc, ee.Reducer.sum(), scale=30, crs=CRS, tileScale=8)
            .map(lambda f: f.setGeometry(f.geometry().centroid(100))))

p = areas_s2(deptos.filter(ee.Filter.eq("link", "82084"))).first().getInfo()["properties"]
y = 2021
print(f"Rosario {y} — máscara 60 m: área {p[f's2_{y}']:,.0f} ha | "
      f"soja fuera {100 * p[f's2_sojafuera_{y}'] / p[f'soja_{y}']:.1f} % | "
      f"máscara no agrícola {100 * (1 - p[f's2_ext_{y}'] / p[f's2_{y}']):.1f} %")

Rosario 2021 — máscara 60 m: área 101,179 ha | soja fuera 10.7 % | máscara no agrícola 5.8 %


In [10]:
tarea = ee.batch.Export.table.toAsset(
    collection=areas_s2(deptos), description="comparacion_mascara_60m_2020_2022",
    assetId=f"{BASE}/comparacion_mascara_60m_2020_2022")
tarea.start()
print("Tarea lanzada:", tarea.status()["state"])

Tarea lanzada: READY


In [11]:
import ee
import pandas as pd
ee.Initialize(project="ee-mnolasco")
BASE = "users/mnolasco/FORMAR_2024_2026"
T = "../results/tables"

feats = ee.FeatureCollection(f"{BASE}/comparacion_mascara_60m_2020_2022").getInfo()["features"]
info = pd.read_csv(f"{T}/seleccion_imagen_2021.csv", dtype={"link": str})[["link", "departamento", "provincia"]]
filas = []
for f in feats:
    p = f["properties"]
    for y in [2020, 2021, 2022]:
        filas.append({"link": str(p["link"]).zfill(5), "campania": y, "mascara": "s2_60m",
                      "ext": p[f"ext_{y}"], "soja": p[f"soja_{y}"], "area": p[f"s2_{y}"],
                      "ext_en": p[f"s2_ext_{y}"], "soja_fuera": p[f"s2_sojafuera_{y}"]})
s2 = pd.DataFrame(filas).merge(info, on="link", how="left")

cm = pd.read_csv(f"{T}/comparacion_mascaras_2020_2025.csv", dtype={"link": str})
cm = cm[cm["campania"].between(2020, 2022) & cm["mascara"].isin(["agri19", "modis"])]
todo = pd.concat([cm, s2], ignore_index=True)
todo.to_csv(f"{T}/comparacion_3mascaras_2020_2022.csv", index=False)
ORD = ["agri19", "modis", "s2_60m"]

def ind(g):
    s = g[["ext", "soja", "area", "ext_en", "soja_fuera"]].sum()
    return pd.Series({"soja_fuera_%": 100 * s["soja_fuera"] / s["soja"],
                      "mascara_no_agri_%": 100 * (1 - s["ext_en"] / s["area"]),
                      "area_vs_INTA_%": 100 * s["area"] / s["ext"]})

print(f"Departamentos: {todo['link'].nunique()} | campañas 2020–2022\n")
print("1. Total de los 95 departamentos:")
print(todo.groupby("mascara").apply(ind).reindex(ORD).round(1).to_string())
print("\n2. Soja fuera de la máscara (%), por campaña:")
print(todo.groupby(["campania", "mascara"]).apply(ind)["soja_fuera_%"].unstack()[ORD].round(1).to_string())
pp = todo.groupby(["provincia", "mascara"]).apply(ind).unstack("mascara")
print("\n3. Soja fuera (%), por provincia:")
print(pp["soja_fuera_%"][ORD].round(1).to_string())
print("\n4. Máscara que no fue agrícola (%), por provincia:")
print(pp["mascara_no_agri_%"][ORD].round(1).to_string())
print("\n5. Área de la máscara respecto del área agrícola INTA (%), por provincia:")
print(pp["area_vs_INTA_%"][ORD].round(1).to_string())

Departamentos: 95 | campañas 2020–2022

1. Total de los 95 departamentos:
         soja_fuera_%  mascara_no_agri_%  area_vs_INTA_%
mascara                                                 
agri19           22.2               11.7            84.6
modis            26.8               13.7            83.8
s2_60m            8.9               10.1            97.5

2. Soja fuera de la máscara (%), por campaña:
mascara   agri19  modis  s2_60m
campania                       
2020        21.2   28.8     8.9
2021        24.5   25.2    10.5
2022        20.6   26.1     7.1

3. Soja fuera (%), por provincia:
mascara       agri19  modis  s2_60m
provincia                          
Buenos Aires    34.7   34.2     9.5
Córdoba          7.3   11.3     8.1
Entre Ríos      17.1   57.9    14.6
La Pampa        58.8   13.6     4.3
Santa Fe        21.9   30.1     8.0

4. Máscara que no fue agrícola (%), por provincia:
mascara       agri19  modis  s2_60m
provincia                          
Buenos Aires    10.2   